# 07. Walking distance to food

This notebook measures, for every inhabited block in the 92 metropolitan areas,
the distance on foot along the street network to the nearest food destination
under each definition of a food destination.

Distance is measured through the pedestrian network rather than in a straight
line. A straight line ignores the motorway, the river and the railway, and in
Mexican cities those features cut neighbourhoods off from retail that is close
in a straight line and far on foot. The network for each area was assembled
separately and holds the ways a pedestrian may use.

Thirteen destination sets are measured. Nine come from the establishment
register and form two branches and a union. The self-service branch runs from
supermarkets alone, to supermarkets with warehouse clubs, to those with small
format self-service, to those with pharmacies holding a grocery section. The
traditional branch runs from the sellers of perishable produce, protein and
dairy to the specialist fresh and staple sellers, and from those to two
extensions that do not contain one another: the fresh set with craft dairy and
the fresh set with neighbourhood grocery stores. The union is every food retail
and food service establishment, and it contains every other set. Four more sets
add street commerce to the perishable set, the fresh set, the fresh set with
grocery stores and the union, because stalls on a block are a food source that
no register holds.

For each set the quantity computed is the distance from the block to the nearest
member of that set, which is the standard proximity measure in work on food
access. It is reported in metres along the network and in minutes at a walking
speed of 4.8 kilometres an hour, the figure conventionally used for pedestrian
accessibility.

Measurement proceeds one metropolitan area at a time. Every way in the network
is divided into pieces of at most fifty metres, and every vertex of the result
becomes a node of a graph whose edges carry the straight length between
consecutive vertices. Dividing the ways matters because the networks were
simplified when they were extracted, so a way between two intersections can run
for several hundred metres, and a block or an outlet standing halfway along it
would otherwise be attached to an intersection far from where it is. Block
points and destinations are attached to the nearest node, and the distance of
that attachment is kept.

The distance a resident walks is then the sum of three parts: from the block
point to its node, along the network to the destination's node, and from that
node to the destination. The second and third parts are found together in one
search per set, by joining an auxiliary source to every destination node at a
cost equal to the shortest attachment among the destinations at that node and
expanding outward from the source, which gives every node in the area its
distance in one pass. The first part is added for each block. Without the
attachment distances, a block and an outlet attached to the same node would be
at a distance of exactly zero however far apart they stand, and the resulting
mass of zeros would distort any measure taken on a logarithmic scale. A block
that carries street commerce on one of its own faces is at zero distance from
the sets that include street commerce, by construction, and that is the only
case in which a distance of zero is assigned.

Three properties of the result are recorded alongside it. The distance from a
block to its node is kept for every block, since a block far from any node is
measured with less precision than one on a street. A block that no destination
can reach through the network has no distance rather than a large one, and the
number of such blocks is reported for each set. And blocks with no inhabitants
are not measured, because there is nobody there whose access is in question,
while such a block still serves as a destination when it carries street
commerce.

Inputs are `Results/04_blocks.parquet` for the frame and its population,
`Results/05_outlets.parquet` for the register destinations,
`Results/06_street_commerce.parquet` for the street layer,
`Results/03_metro_boundaries.gpkg` for the area boundaries,
`Inputs/MGN/` for the block geometry that block points are taken from, and
`Networks/pedestrian_networks/` for the 92 pedestrian networks.

Two further kinds of destination set are measured for the robustness checks
and are kept apart from the thirteen definitions. The first removes from the
supermarket set and from the fresh set every establishment whose coordinate is
shared by ten or more records in the register, since such a coordinate is
usually the centroid of a locality rather than a shop. The second serves as a
placebo for the spatial model estimated later: the fresh set thinned at random
to as many locations as the supermarket set has in the same area, and random
nodes of the network at the same count, each drawn ten times. If the spatial
pattern found for supermarkets appears for a thinned fresh set as well, it
belongs to the spacing of the destinations rather than to what they are.

Output is `Results/07_block_access.parquet`, one row per inhabited block, with
the distance in metres and the time in minutes to the nearest destination in
each set, the distance from the block to its network node, the projected
coordinates of the block, and its population, and
`Results/07_block_access_robustness.parquet`, one row per inhabited block with
the distance to each robustness and placebo set.

Runtime is two to three hours. Each area is cached as it completes, so an
interrupted run resumes where it stopped.

## 1. Setup

In [ ]:
import sys, os, glob, zipfile, shutil, hashlib, importlib, time

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT = os.environ.get('MXO_BASE', '/content/drive/MyDrive/Mexico_Obesity')
os.environ['MXO_BASE'] = PROJECT
sys.path.insert(0, f'{PROJECT}/Code')
import mxo_utils as mx
importlib.reload(mx)

assert mx.BASE == PROJECT, f'the module resolves to {mx.BASE}'

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import pyogrio
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix, csr_matrix
from scipy.sparse.csgraph import dijkstra

rep = mx.Report('07_block_access').start()
mx.flow_reset()

INPUTS = mx.DIRS['inputs']
RESULTS = mx.DIRS['results']

WALK_KMH = 4.8
WALK_M_PER_MIN = WALK_KMH * 1000 / 60
DENSIFY_M = 50.0         # longest piece a way is divided into before nodes are taken
NODE_PRECISION = 2       # decimal places of a metre when matching vertices
MIN_EDGE_M = 1e-6        # an edge or an attachment shorter than this is given this length
print(f'walking speed {WALK_KMH} km/h, {WALK_M_PER_MIN:.1f} metres a minute')
print(f'ways divided into pieces of at most {DENSIFY_M:.0f} m')

## 2. The frame, the destinations and the networks

In [ ]:
BLOCKS = pd.read_parquet(os.path.join(RESULTS, '04_blocks.parquet'),
                         columns=['CVEGEO_MZA', 'CVEGEO', 'slug',
                                  'macro_region', 'POBTOT'])
STREET = pd.read_parquet(os.path.join(RESULTS, '06_street_commerce.parquet'),
                         columns=['CVEGEO_MZA', 'street_any', 'street_known'])
BLOCKS = BLOCKS.merge(STREET, on='CVEGEO_MZA', how='left')
BLOCKS['street_any'] = BLOCKS['street_any'].fillna(False).astype(bool)
BLOCKS['street_known'] = BLOCKS['street_known'].fillna(False).astype(bool)
BLOCKS['inhabited'] = BLOCKS['POBTOT'] > 0

OUT = pd.read_parquet(os.path.join(RESULTS, '05_outlets.parquet'))

METRO = gpd.read_file(os.path.join(RESULTS, '03_metro_boundaries.gpkg'),
                      layer='metro_boundaries')

NETDIR = next(d for d in mx.NETWORKS_CANDIDATES if os.path.isdir(d))
files = sorted(glob.glob(os.path.join(NETDIR, '*.gpkg')))
assert files, f'no networks in {NETDIR}'

# Each network file is assigned to the area whose name appears in its file name,
# taking the longest name that appears. File names may therefore carry a prefix
# or a suffix such as the mode of travel or the projection, and an area whose
# name is contained in a longer area name is not mistaken for it.
AREAS = sorted(METRO['slug'], key=len, reverse=True)
STEMS = {f: mx._slug_name(
    [os.path.splitext(os.path.basename(f))[0]]).iloc[0] for f in files}

NETS, ORPHAN, CLASH = {}, [], []
for f in files:
    hit = next((a for a in AREAS if a in STEMS[f]), None)
    if hit is None:
        ORPHAN.append(f)
    elif hit in NETS:
        CLASH.append((hit, NETS[hit], f))
    else:
        NETS[hit] = f

ABSENT = sorted(set(METRO['slug']) - set(NETS))
if ABSENT or ORPHAN or CLASH:
    print(f'{len(files)} files in {NETDIR}\n')
    print('file name slugs:')
    for f in files:
        print(f'   {os.path.basename(f):50s} -> {STEMS[f]}')
    print(f'\nareas with no file ({len(ABSENT)}): {ABSENT}')
    print(f'files matching no area ({len(ORPHAN)}): '
          f'{[os.path.basename(f) for f in ORPHAN]}')
    print(f'areas matched by more than one file ({len(CLASH)}): '
          f'{[(a, os.path.basename(x), os.path.basename(y)) for a, x, y in CLASH]}')
    raise RuntimeError('the network files do not correspond one to one with '
                       'the 92 areas; the listing above shows which')

print(f'{len(BLOCKS):,} blocks, {int(BLOCKS.inhabited.sum()):,} inhabited, '
      f'{BLOCKS.POBTOT.sum():,.0f} people')
print(f'{len(OUT):,} located establishments')
print(f'{len(METRO)} boundaries, {len(NETS)} networks, one for each area')
print(f'networks folder: {NETDIR}\n')
print('the first five file names and the areas they carry:')
for f in files[:5]:
    print(f'   {os.path.basename(f):50s} -> '
          f'{next(a for a, g in NETS.items() if g == f)}')
rep.add_table('network file for each area', pd.DataFrame(
    {'slug': list(NETS), 'file': [os.path.basename(v) for v in NETS.values()]}),
    max_rows=92)

## 3. The destination sets

Nine sets are columns of the establishment file. Four more join street commerce
to four of them, and their street members are blocks rather than
establishments, so they are assembled inside each area from the blocks of that
area.

In [ ]:
REGISTER_SETS = ['supermarket_only', 'supermarket_club', 'supermarket_mini',
                 'supermarket_mini_pharmacy', 'fresh_perishable', 'fresh_only',
                 'fresh_counter', 'fresh_abarrotes', 'all_food']
STREET_SETS = {'fresh_perishable_street': 'fresh_perishable',
               'fresh_street': 'fresh_only',
               'fresh_abarrotes_street': 'fresh_abarrotes',
               'all_food_street': 'all_food'}
SETS = REGISTER_SETS + list(STREET_SETS)

# The set each set extends. Every pair here is checked for containment of the
# distances at the end, and every register set is also checked against the
# union.
PARENT = {'supermarket_club': 'supermarket_only',
          'supermarket_mini': 'supermarket_club',
          'supermarket_mini_pharmacy': 'supermarket_mini',
          'fresh_only': 'fresh_perishable',
          'fresh_counter': 'fresh_only',
          'fresh_abarrotes': 'fresh_only',
          'fresh_perishable_street': 'fresh_perishable',
          'fresh_street': 'fresh_only',
          'fresh_abarrotes_street': 'fresh_abarrotes',
          'all_food_street': 'all_food'}
NESTED_PAIRS = ([(p, s) for s, p in PARENT.items()]
                + [(s, 'all_food') for s in REGISTER_SETS if s != 'all_food']
                + [('fresh_street', 'fresh_abarrotes_street'),
                   ('fresh_perishable_street', 'fresh_street'),
                   ('fresh_abarrotes_street', 'all_food_street')])

for s in REGISTER_SETS:
    assert s in OUT.columns, s

# The cache is keyed by the networks, the sets measured and the parameters of
# the graph, so a change to any of them yields a different directory.
KEY = hashlib.blake2b(('|'.join(SETS)
                       + f'|{WALK_KMH}|{NODE_PRECISION}|{DENSIFY_M}|snap')
                      .encode(), digest_size=4).hexdigest()
CACHE = mx.cache_dir(f'cache_07_access_{KEY}')
print('cache directory:', CACHE)

sizes = pd.DataFrame({'destination set': REGISTER_SETS,
                      'establishments': [int(OUT[s].sum())
                                         for s in REGISTER_SETS]})
print()
print(sizes.to_string(index=False))

street_blocks = BLOCKS['street_any'] & BLOCKS['street_known']
print(f'\nblocks carrying street commerce, usable as destinations: '
      f'{int(street_blocks.sum()):,}')
print(f'blocks whose street commerce is unknown, excluded as destinations: '
      f'{int((~BLOCKS.street_known).sum()):,}')
rep.add_table('register destination set sizes', sizes, max_rows=10)

## 4. Block geometry

A block point is the representative point of the block polygon, which lies
inside the polygon for every shape, including the long thin and the concave
ones where a centroid can fall outside.

The national block layer is a shapefile inside a compressed archive. Reading
geometry by bounding box needs the spatial index that sits beside the shapefile
in the archive, and that index is not available while the file is read from
inside the archive, so the shapefile and its companions are extracted to local
storage once and read from there.

In [ ]:
LOCAL = '/content/mgn_blocks'
os.makedirs(LOCAL, exist_ok=True)


def locate_block_layer():
    """The archive member holding block polygons keyed by a 16 character key."""
    for arch in sorted(glob.glob(os.path.join(INPUTS, 'MGN', '**', '*.zip'),
                                 recursive=True)):
        with zipfile.ZipFile(arch) as zf:
            shps = [n for n in zf.namelist() if n.lower().endswith('.shp')]
        for shp in shps:
            try:
                info = pyogrio.read_info(f'/vsizip/{arch}/{shp}')
            except Exception:
                continue
            fields = list(info['fields'])
            if 'CVEGEO' not in fields:
                continue
            if 'Polygon' not in str(info['geometry_type']):
                continue
            head = pyogrio.read_dataframe(f'/vsizip/{arch}/{shp}',
                                          columns=['CVEGEO'], max_features=5,
                                          read_geometry=False)
            if head['CVEGEO'].astype(str).str.len().eq(16).all():
                return arch, shp, info
    raise RuntimeError('no block layer found under Inputs/MGN')


ARCH, SHP, INFO = locate_block_layer()
print('archive :', os.path.basename(ARCH))
print('member  :', SHP)
print('features:', f"{INFO['features']:,}")
print('crs     :', str(INFO['crs']))

stem = os.path.splitext(os.path.basename(SHP))[0]
BLOCK_PATH = os.path.join(LOCAL, stem + '.shp')
if not os.path.exists(BLOCK_PATH):
    with mx.stage('extract the block layer'):
        with zipfile.ZipFile(ARCH) as zf:
            for n in zf.namelist():
                if os.path.splitext(os.path.basename(n))[0] == stem:
                    with zf.open(n) as fh, \
                         open(os.path.join(LOCAL, os.path.basename(n)),
                              'wb') as out:
                        shutil.copyfileobj(fh, out)

have = sorted(os.path.splitext(f)[1].lower() for f in os.listdir(LOCAL))
print('\nextracted  :', ' '.join(have))
print('spatial index present:', any(e in have for e in ('.sbn', '.qix')))
BLOCK_CRS = pyogrio.read_info(BLOCK_PATH)['crs']

## 5. One metropolitan area

The steps below are applied to each area in turn.

The network is read and reduced to a graph. Every way is divided into pieces of
at most fifty metres, every vertex of the result becomes a node, matched to the
others by coordinate, and every pair of consecutive vertices becomes an edge
carrying the straight length between them. Ways are undirected, since a
pedestrian may walk either way along a street, and where two edges join the
same pair of nodes the shorter is kept.

Block points and destinations are attached to the nearest node by a spatial
search, and the distance of that attachment is recorded. Since no vertex is more
than fifty metres from the next, the attachment distance is close to the
distance from the point to the street.

The distance from every node to the nearest destination, counting the
destination's own attachment, is found by one search. An auxiliary node is
joined to every destination node by an edge whose length is the shortest
attachment among the destinations at that node, and the search starts from the
auxiliary node. The block's own attachment is then added. One search per set
returns the distance for every block in the area, which is what makes thirteen
sets across 92 areas tractable.

In [ ]:
def build_graph(path):
    """Nodes, their coordinates, and the weighted adjacency of a network.

    The adjacency has one row and column more than there are nodes. The last
    index is the auxiliary source used by the search, and it carries no edges
    here.
    """
    layers = [name for name, kind in pyogrio.list_layers(path)
              if 'Line' in str(kind)]
    if not layers:
        raise RuntimeError(f'{os.path.basename(path)} holds no line layer')
    name = 'edges' if 'edges' in layers else layers[0]
    E = pyogrio.read_dataframe(path, layer=name, columns=[])
    E = E[E.geometry.notna() & ~E.geometry.is_empty]
    E = E.explode(index_parts=False, ignore_index=True)
    # A way with a single vertex or no length cannot be walked along and
    # cannot be divided, so it is left out before the division.
    E = E[(E.geometry.geom_type == 'LineString') & (E.geometry.length > 0)]
    n_ways = len(E)
    net_len = float(E.geometry.length.sum())

    # Every way is divided into pieces of at most DENSIFY_M, and every vertex
    # becomes a node. Consecutive vertices of one way form an edge.
    geom = shapely.segmentize(E.geometry.values, DENSIFY_M)
    pts, which = shapely.get_coordinates(geom, return_index=True)
    same = which[:-1] == which[1:]
    p0, p1 = pts[:-1][same], pts[1:][same]
    seg = np.hypot(p1[:, 0] - p0[:, 0], p1[:, 1] - p0[:, 1])
    keep = seg > 0
    p0, p1, seg = p0[keep], p1[keep], seg[keep]

    stacked = np.round(np.vstack([p0, p1]), NODE_PRECISION)
    coords, inverse = np.unique(stacked, axis=0, return_inverse=True)
    inverse = np.asarray(inverse).ravel()
    half = len(p0)
    u, v = inverse[:half], inverse[half:]

    ed = pd.DataFrame({'u': np.minimum(u, v), 'v': np.maximum(u, v),
                       'w': np.maximum(seg, MIN_EDGE_M)})
    ed = ed[ed.u != ed.v].groupby(['u', 'v'], sort=False)['w'].min().reset_index()

    n = len(coords)
    W = coo_matrix((np.concatenate([ed.w.values, ed.w.values]),
                    (np.concatenate([ed.u.values, ed.v.values]),
                     np.concatenate([ed.v.values, ed.u.values]))),
                   shape=(n + 1, n + 1)).tocsr()
    return coords, W, E.crs, n_ways, net_len


def nearest_with_offsets(W, n, dest_nodes, dest_offsets):
    """Distance from every node to the nearest destination node, each
    destination counted with its own attachment distance.

    An auxiliary source, index n, is joined to each destination node by an edge
    of length equal to the shortest attachment among the destinations at that
    node. One directed search from the source then gives, for every node, the
    length of the shortest route to any destination plus that destination's
    attachment. The adjacency is stored in both directions, so the search is
    run as directed and the auxiliary edges are used in one direction only.
    """
    order = np.argsort(dest_offsets, kind='stable')
    nodes, first = np.unique(dest_nodes[order], return_index=True)
    offsets = np.maximum(dest_offsets[order][first], MIN_EDGE_M)
    aux = csr_matrix((offsets, (np.full(len(nodes), n), nodes)),
                     shape=(n + 1, n + 1))
    d = dijkstra(W + aux, directed=True, indices=n)
    return d[:n]


def block_points(slug, boundary, keys):
    """Representative points of the frame blocks of one area."""
    bounds = (gpd.GeoSeries([boundary], crs=METRO.crs)
                .to_crs(BLOCK_CRS).total_bounds)
    G = pyogrio.read_dataframe(BLOCK_PATH, columns=['CVEGEO'], bbox=tuple(bounds))
    G = G[G['CVEGEO'].isin(keys)]
    G = G[G.geometry.notna() & ~G.geometry.is_empty]
    G = G.drop_duplicates('CVEGEO')
    return gpd.GeoDataFrame({'CVEGEO_MZA': G['CVEGEO'].values},
                            geometry=G.geometry.representative_point().values,
                            crs=G.crs)

In [ ]:
def attach(slug, boundary):
    """The graph of one area with its blocks and destinations attached."""
    frame = BLOCKS[BLOCKS.slug == slug]
    coords, W, net_crs, n_ways, net_len = build_graph(NETS[slug])
    tree = cKDTree(coords)

    pts = block_points(slug, boundary, set(frame.CVEGEO_MZA))
    pts = pts.to_crs(net_crs)
    pts = pts.merge(frame[['CVEGEO_MZA', 'POBTOT', 'inhabited', 'street_any',
                           'street_known']], on='CVEGEO_MZA')
    bxy = np.column_stack([pts.geometry.x.values, pts.geometry.y.values])
    bsnap, bnode = tree.query(bxy, workers=-1)

    dest = OUT[OUT.slug == slug]
    dgeo = (gpd.GeoSeries(gpd.points_from_xy(dest.x_m, dest.y_m),
                          crs=f'EPSG:{mx.CRS_METRIC}').to_crs(net_crs))
    dxy = np.column_stack([dgeo.x.values, dgeo.y.values])
    dsnap, dnode = tree.query(dxy, workers=-1)
    return dict(frame=frame, coords=coords, W=W, net_crs=net_crs,
                n_ways=n_ways, net_len=net_len, pts=pts, bsnap=bsnap,
                bnode=bnode, dest=dest, dsnap=dsnap, dnode=dnode)


def measure(slug, boundary):
    """Distance from each inhabited block of one area to each destination set."""
    a = attach(slug, boundary)
    frame, coords, W, net_crs = a['frame'], a['coords'], a['W'], a['net_crs']
    n_ways, net_len, pts = a['n_ways'], a['net_len'], a['pts']
    bsnap, bnode, dest, dsnap, dnode = (a['bsnap'], a['bnode'], a['dest'],
                                        a['dsnap'], a['dnode'])
    n = len(coords)

    # Street commerce destinations are the blocks that carry it, attached by
    # their own block point, so their attachment distance is the block's.
    is_street = (pts.street_any & pts.street_known).values
    street_node, street_snap = bnode[is_street], bsnap[is_street]
    own_street = is_street & pts.inhabited.values

    metric = pts.to_crs(epsg=mx.CRS_METRIC)
    out = {'CVEGEO_MZA': pts.CVEGEO_MZA.values, 'POBTOT': pts.POBTOT.values,
           'snap_m': bsnap,
           'x_m': metric.geometry.x.values, 'y_m': metric.geometry.y.values}
    reach = {}
    for s in SETS:
        if s in REGISTER_SETS:
            m = dest[s].values
            nodes, offsets = dnode[m], dsnap[m]
        else:
            m = dest[STREET_SETS[s]].values
            nodes = np.concatenate([dnode[m], street_node])
            offsets = np.concatenate([dsnap[m], street_snap])
        if len(nodes) == 0:
            out[f'distance_m_{s}'] = np.full(len(pts), np.nan)
            reach[s] = 0
            continue
        d = nearest_with_offsets(W, n, nodes, offsets)
        v = d[bnode] + bsnap
        v[np.isinf(d[bnode])] = np.nan
        if s in STREET_SETS:
            # A block with stalls on one of its own faces is at the source.
            v[is_street] = 0.0
        out[f'distance_m_{s}'] = v
        reach[s] = len(np.unique(nodes))

    R = pd.DataFrame(out)
    R['slug'] = slug
    R = R[pts.inhabited.values].reset_index(drop=True)
    meta = {'slug': slug, 'ways': n_ways, 'nodes': n,
            'edges': int(W.nnz // 2),
            'network_km': round(net_len / 1000, 1),
            'blocks': int(pts.inhabited.sum()),
            'blocks_found': len(pts), 'blocks_in_frame': len(frame),
            'establishments': len(dest),
            'street_destinations': int(len(np.unique(street_node))),
            'block_snap_median_m': round(float(np.median(bsnap)), 1),
            'block_snap_p99_m': round(float(np.percentile(bsnap, 99)), 1),
            'dest_snap_median_m': round(float(np.median(dsnap)), 1)
                                  if len(dsnap) else np.nan,
            'dest_snap_p99_m': round(float(np.percentile(dsnap, 99)), 1)
                               if len(dsnap) else np.nan}
    meta.update({f'reachable_{s}': reach[s] for s in SETS})
    return R, meta

## 6. Every metropolitan area

In [ ]:
parts, metas = [], []
order = (BLOCKS.groupby('slug')['POBTOT'].sum()
               .sort_values(ascending=False).index.tolist())
geom = METRO.set_index('slug').geometry

t0 = time.time()
for i, slug in enumerate(order, 1):
    fp = os.path.join(CACHE, f'{slug}.parquet')
    mp = os.path.join(CACHE, f'{slug}_meta.parquet')
    if os.path.exists(fp) and os.path.exists(mp):
        parts.append(pd.read_parquet(fp))
        metas.append(pd.read_parquet(mp))
        continue
    with mx.stage(f'[{i:>2}/{len(order)}] {slug}'):
        R, meta = measure(slug, geom.loc[slug])
        R.to_parquet(fp, index=False)
        M = pd.DataFrame([meta])
        M.to_parquet(mp, index=False)
    parts.append(R)
    metas.append(M)

A = pd.concat(parts, ignore_index=True)
A = A.merge(BLOCKS[['CVEGEO_MZA', 'macro_region']], on='CVEGEO_MZA', how='left')
NET = pd.concat(metas, ignore_index=True)
print(f'\n{len(A):,} inhabited blocks measured in {time.time() - t0:.0f}s')

for s in SETS:
    A[f'walk_min_{s}'] = (A[f'distance_m_{s}'] / WALK_M_PER_MIN).round(2)


def wq(frame, col, q, weight='POBTOT'):
    """Weighted quantile of col, over the rows where col is present."""
    f = frame[[col, weight]].dropna()
    if not len(f):
        return np.nan
    f = f.sort_values(col)
    cw = f[weight].cumsum() / f[weight].sum()
    return float(f.loc[cw >= q, col].iloc[0])

## 7. Robustness and placebo destination sets

Two further kinds of destination set are measured here and written to a
separate file, so that the thirteen definitions above stay as they are.

The first removes every establishment standing on a coordinate shared by ten
or more records of the register. Such a coordinate is usually the centroid of
a locality or a municipality rather than the position of a shop, although some
are real public markets with many stalls. The supermarket set and the fresh
set are each measured again without those records.

The second is a placebo for the spatial model estimated later, in which the
distance to a destination is related to the deprivation of the tract and to the
mean deprivation of its neighbours. Supermarkets are sparse, about one for
every seven tracts, so the distance to the nearest one changes slowly from one
tract to the next, and a slowly changing outcome will tend to attach itself to
the slowly changing regressor whatever the destinations are. To see how much of
the pattern is spacing alone, the fresh set is thinned at random in each area
to as many network locations as the supermarket set has there, and the
distance to the thinned set is measured. Random nodes of the network, at the
same count, give the reference for a set of destinations that has no relation
to deprivation at all. Each is drawn ten times, with a seed fixed by the name
of the area so that the draws do not depend on the order in which areas run.

In [ ]:
R_DRAWS = 10
SHARED_MIN = 10          # records on one coordinate that mark it as shared
ROB_SETS = (['supermarket_club_noshared', 'fresh_only_noshared']
            + [f'fresh_thinned_{r}' for r in range(R_DRAWS)]
            + [f'random_nodes_{r}' for r in range(R_DRAWS)])
ROB_KEY = hashlib.blake2b((KEY + f'|robust|{R_DRAWS}|{SHARED_MIN}').encode(),
                          digest_size=4).hexdigest()
ROB_CACHE = mx.cache_dir(f'cache_07_robust_{ROB_KEY}')
print('cache directory:', ROB_CACHE)


def unique_nodes(nodes, offsets):
    """Each destination node once, with the shortest attachment at it."""
    order = np.argsort(offsets, kind='stable')
    u, first = np.unique(nodes[order], return_index=True)
    return u, offsets[order][first]


def measure_robustness(slug, boundary):
    a = attach(slug, boundary)
    W, coords, pts = a['W'], a['coords'], a['pts']
    bsnap, bnode, dest, dsnap, dnode = (a['bsnap'], a['bnode'], a['dest'],
                                        a['dsnap'], a['dnode'])
    n = len(coords)
    seed = int.from_bytes(hashlib.blake2b(slug.encode(), digest_size=4)
                          .digest(), 'little')
    rng = np.random.default_rng([20201105, seed])
    shared = dest['point_records'].to_numpy() >= SHARED_MIN

    out = {'CVEGEO_MZA': pts.CVEGEO_MZA.values, 'POBTOT': pts.POBTOT.values}

    def put(name, nodes, offsets):
        if len(nodes) == 0:
            out[f'distance_m_{name}'] = np.full(len(pts), np.nan)
            return
        d = nearest_with_offsets(W, n, nodes, offsets)
        v = d[bnode] + bsnap
        v[np.isinf(d[bnode])] = np.nan
        out[f'distance_m_{name}'] = v

    for base in ['supermarket_club', 'fresh_only']:
        m = dest[base].values & ~shared
        put(f'{base}_noshared', dnode[m], dsnap[m])

    sup_nodes, _ = unique_nodes(dnode[dest['supermarket_club'].values],
                                dsnap[dest['supermarket_club'].values])
    fr_nodes, fr_off = unique_nodes(dnode[dest['fresh_only'].values],
                                    dsnap[dest['fresh_only'].values])
    k = len(sup_nodes)
    for r in range(R_DRAWS):
        if k == 0 or len(fr_nodes) == 0:
            put(f'fresh_thinned_{r}', np.array([], int), np.array([]))
            put(f'random_nodes_{r}', np.array([], int), np.array([]))
            continue
        pick = rng.choice(len(fr_nodes), size=min(k, len(fr_nodes)),
                          replace=False)
        put(f'fresh_thinned_{r}', fr_nodes[pick], fr_off[pick])
        nodes = rng.choice(n, size=min(k, n), replace=False)
        put(f'random_nodes_{r}', nodes, np.zeros(len(nodes)))

    R = pd.DataFrame(out)
    R['slug'] = slug
    R = R[pts.inhabited.values].reset_index(drop=True)
    meta = {'slug': slug, 'supermarket_nodes': int(k),
            'fresh_nodes': int(len(fr_nodes)),
            'supermarkets_on_shared_points':
                int((dest['supermarket_club'].values & shared).sum()),
            'fresh_on_shared_points':
                int((dest['fresh_only'].values & shared).sum())}
    return R, meta


parts, metas = [], []
t0 = time.time()
for i, slug in enumerate(order, 1):
    fp = os.path.join(ROB_CACHE, f'{slug}.parquet')
    mp = os.path.join(ROB_CACHE, f'{slug}_meta.parquet')
    if os.path.exists(fp) and os.path.exists(mp):
        parts.append(pd.read_parquet(fp))
        metas.append(pd.read_parquet(mp))
        continue
    with mx.stage(f'[{i:>2}/{len(order)}] {slug}'):
        R, meta = measure_robustness(slug, geom.loc[slug])
        R.to_parquet(fp, index=False)
        M = pd.DataFrame([meta])
        M.to_parquet(mp, index=False)
    parts.append(R)
    metas.append(M)
ROB = pd.concat(parts, ignore_index=True)
ROBMETA = pd.concat(metas, ignore_index=True)
print(f'\n{len(ROB):,} inhabited blocks measured against the robustness sets '
      f'in {time.time() - t0:.0f}s')
assert len(ROB) == len(A) and set(ROB.CVEGEO_MZA) == set(A.CVEGEO_MZA)

print(f'\nrecords removed as standing on a shared coordinate: '
      f'{int(ROBMETA.supermarkets_on_shared_points.sum()):,} supermarkets, '
      f'{int(ROBMETA.fresh_on_shared_points.sum()):,} fresh sellers')
rows = []
for s in ['supermarket_club', 'supermarket_club_noshared', 'fresh_only',
          'fresh_only_noshared']:
    src = A if s in ('supermarket_club', 'fresh_only') else ROB
    col = f'distance_m_{s}'
    rows.append({'destination set': s,
                 'median metres': round(wq(src, col, 0.5)),
                 'p90 metres': round(wq(src, col, 0.9)),
                 'blocks without a distance': int(src[col].isna().sum())})
for name, pref in [('fresh thinned to supermarket count', 'fresh_thinned_'),
                   ('random network nodes', 'random_nodes_')]:
    meds = [wq(ROB, f'distance_m_{pref}{r}', 0.5) for r in range(R_DRAWS)]
    rows.append({'destination set': f'{name}, mean of {R_DRAWS} draws',
                 'median metres': round(float(np.mean(meds))),
                 'p90 metres': round(float(np.mean(
                     [wq(ROB, f'distance_m_{pref}{r}', 0.9)
                      for r in range(R_DRAWS)]))),
                 'blocks without a distance': int(np.mean(
                     [ROB[f'distance_m_{pref}{r}'].isna().sum()
                      for r in range(R_DRAWS)]))})
ROBDIST = pd.DataFrame(rows)
print()
print(ROBDIST.to_string(index=False))
rep.add_table('robustness and placebo sets, distance to the nearest '
              'destination', ROBDIST, max_rows=10)
rep.add_table('robustness sets by area', ROBMETA, max_rows=92)

## 8. Networks and attachment

The size of each network is reported after division into pieces, with the
distance from its blocks and its destinations to the nodes they were attached
to. A median attachment of a few metres means the blocks sit on the streets of
the network. A large one means the network is sparse where the population is,
and the distances for that area carry a corresponding uncertainty. Since the
attachment now enters the measure, a large attachment lengthens the measured
distance rather than silently shortening it.

In [ ]:
cols = ['slug', 'ways', 'nodes', 'edges', 'network_km', 'blocks',
        'establishments', 'street_destinations', 'block_snap_median_m',
        'block_snap_p99_m', 'dest_snap_median_m', 'dest_snap_p99_m']
print(NET[cols].sort_values('blocks', ascending=False).head(25)
         .to_string(index=False))
rep.add_table('networks and attachment by area', NET, max_rows=92)

print(f'\nnodes per kilometre of network, over all areas: '
      f'{NET.nodes.sum() / NET.network_km.sum():.1f}')

print('\ndistance from a block to its node, over all blocks:\n')
print(A['snap_m'].describe(percentiles=[0.5, 0.9, 0.99]).round(1).to_string())

print('\ndistance from a block to its node, weighted by population:\n')
for q in (0.5, 0.9, 0.99):
    print(f'   p{int(q * 100):<3d} {wq(A, "snap_m", q):,.1f} m')

print('\nareas whose blocks sit furthest from their networks:\n')
print(NET.nlargest(10, 'block_snap_median_m')[
    ['slug', 'blocks', 'nodes', 'block_snap_median_m', 'block_snap_p99_m']]
    .to_string(index=False))

lost = NET['blocks_in_frame'] - NET['blocks_found']
if int(lost.sum()):
    print(f'\nframe blocks without geometry in the national layer: '
          f'{int(lost.sum()):,}')
    print(NET.assign(missing=lost).nlargest(10, 'missing')[
        ['slug', 'blocks_in_frame', 'blocks_found', 'missing']]
        .to_string(index=False))

## 9. Blocks no destination can reach

A block in a part of the network that no destination can reach through the
network has no distance to that set. This happens where a network breaks into
pieces, which it does along a motorway or a river, or at the edge of an area.
The count and the population are reported for each set, and the measure is
missing for those blocks rather than set to a large number.

In [ ]:
rows = []
for s in SETS:
    bad = A[f'distance_m_{s}'].isna()
    rows.append({'destination set': s,
                 'blocks without a distance': int(bad.sum()),
                 'share of blocks': round(float(bad.mean()), 5),
                 'population': int(A.loc[bad, 'POBTOT'].sum()),
                 'share of population': round(
                     float(A.loc[bad, 'POBTOT'].sum() / A.POBTOT.sum()), 5)})
UNREACH = pd.DataFrame(rows)
print(UNREACH.to_string(index=False))
rep.add_table('blocks no destination can reach', UNREACH, max_rows=10)

worst = (A.assign(bad=A[f'distance_m_{SETS[0]}'].isna())
          .groupby('slug').agg(blocks=('bad', 'size'), unreached=('bad', 'sum'))
          .assign(share=lambda d: (d.unreached / d.blocks).round(4))
          .nlargest(10, 'share'))
print(f'\nareas with the largest share unreached, set {SETS[0]}:\n')
print(worst.to_string())

## 10. Distance by destination set

The distance to the nearest destination is summarised over people rather than
over blocks, since a block holding two thousand people and one holding twenty
are not equal observations. The population weighted median is the distance of
the median resident.

Two thresholds are reported. Five hundred metres is the catchment conventionally
treated as walkable in work on pedestrian access. One thousand six hundred and
nine metres is one mile, the distance at which an urban area with no supermarket
is classified as a food desert in United States federal practice, and it is
reported here so that the Mexican figures can be read against that standard.

In [ ]:
THRESHOLDS = [500, 1609]
rows = []
for s in SETS:
    col = f'distance_m_{s}'
    f = A[[col, 'POBTOT']].dropna()
    pop = f.POBTOT.sum()
    r = {'destination set': s,
         'median metres': round(wq(A, col, 0.5)),
         'p25 metres': round(wq(A, col, 0.25)),
         'p75 metres': round(wq(A, col, 0.75)),
         'p90 metres': round(wq(A, col, 0.90)),
         'median minutes': round(wq(A, col, 0.5) / WALK_M_PER_MIN, 1)}
    for t in THRESHOLDS:
        r[f'share within {t} m'] = round(
            float(f.loc[f[col] <= t, 'POBTOT'].sum() / pop), 4)
    rows.append(r)
DIST = pd.DataFrame(rows)
print(DIST.to_string(index=False))
rep.add_table('distance to the nearest destination by set', DIST, max_rows=10)

# A distance of exactly zero arises only for a block with street commerce on
# its own faces, under the sets that include street commerce. The count is
# reported for every set so that the absence of any other mass at zero can be
# seen.
print('\n--- blocks at a distance of exactly zero ---\n')
rows = []
for s in SETS:
    col = f'distance_m_{s}'
    z = A[col] == 0
    rows.append({'destination set': s, 'blocks at zero': int(z.sum()),
                 'share of blocks': round(float(z.mean()), 5),
                 'population at zero': int(A.loc[z, 'POBTOT'].sum()),
                 'blocks under 10 m': int((A[col] < 10).sum())})
ZERO = pd.DataFrame(rows)
print(ZERO.to_string(index=False))
rep.add_table('blocks at zero distance by set', ZERO, max_rows=16)

In [ ]:
print('--- population weighted median metres, by official classification ---\n')
metros = mx.load_metros()
tipo = metros.assign(slug=mx._slug_name(metros['metro_key']).values)[
    ['slug', 'metro_name', 'tipo']]
AT = A.merge(tipo, on='slug', how='left')
assert AT['tipo'].notna().all()

rows = []
for t in mx.TIPO_ORDER:
    sub = AT[AT.tipo == t]
    r = {'tipo': t, 'areas': sub.slug.nunique(), 'blocks': len(sub),
         'population': int(sub.POBTOT.sum())}
    r.update({s: round(wq(sub, f'distance_m_{s}', 0.5)) for s in SETS})
    rows.append(r)
print(pd.DataFrame(rows).to_string(index=False))
rep.add_table('median distance by official classification',
              pd.DataFrame(rows), max_rows=10)

print('\n--- by macro region ---\n')
rows = []
for g, sub in A.groupby('macro_region'):
    r = {'macro_region': g, 'blocks': len(sub),
         'population': int(sub.POBTOT.sum())}
    r.update({s: round(wq(sub, f'distance_m_{s}', 0.5)) for s in SETS})
    rows.append(r)
print(pd.DataFrame(rows).to_string(index=False))
rep.add_table('median distance by macro region', pd.DataFrame(rows),
              max_rows=10)

In [ ]:
print('--- the span between the narrowest and the widest definition ---\n')
span = DIST.set_index('destination set')[['median metres', 'median minutes',
                                          'share within 500 m']]
span['relative to the widest'] = (
    span['median metres'] / span.loc['all_food_street', 'median metres']
    ).round(2)
print(span.to_string())
rep.add_table('span across the definitions', span.reset_index(), max_rows=10)

print('\n--- by area, the widest and narrowest definitions ---\n')
rows = []
for slug, sub in A.groupby('slug'):
    rows.append({'slug': slug, 'blocks': len(sub),
                 'population': int(sub.POBTOT.sum()),
                 **{s: round(wq(sub, f'distance_m_{s}', 0.5)) for s in SETS}})
BYAREA = pd.DataFrame(rows).merge(tipo, on='slug')
print(BYAREA.nlargest(15, 'supermarket_club')[
    ['metro_name', 'tipo', 'population', 'supermarket_club', 'fresh_only',
     'fresh_street', 'all_food', 'all_food_street']].to_string(index=False))
rep.add_table('median distance by area', BYAREA, max_rows=92)

## 11. Checks

Each check names the condition under which it fails.

In [ ]:
print('--- checks ---\n')

d = int(A['CVEGEO_MZA'].duplicated().sum())
print(f'1. duplicate blocks                          {d}')
assert d == 0

# Every inhabited block of the frame has to be measured, or its population is
# absent from every result that follows.
expected = int(BLOCKS.inhabited.sum())
print(f'2. blocks measured against inhabited blocks   {len(A):,} '
      f'against {expected:,}')

# An area absent here would be absent from every later result.
gone = sorted(set(METRO['slug']) - set(A['slug']))
print(f'3. areas with no measured block              {len(gone)}')
assert not gone, gone

# A wider set contains every destination of a narrower one, each with the same
# attachment, so its distance cannot be greater.
viol = []
for narrow, wide in NESTED_PAIRS:
    a, b = A[f'distance_m_{narrow}'], A[f'distance_m_{wide}']
    bad = int((b > a + 1e-6).sum())
    viol.append({'narrower': narrow, 'wider': wide, 'blocks violating': bad})
V = pd.DataFrame(viol)
print(f'4. containment between the sets')
print(V.to_string(index=False))
assert V['blocks violating'].sum() == 0
rep.add_table('containment between the destination sets', V, max_rows=20)

# A distance cannot be negative.
neg = int(sum((A[f'distance_m_{s}'] < 0).sum() for s in SETS))
print(f'5. negative distances                        {neg}')
assert neg == 0

# A block on its own street commerce is at zero distance from every set that
# includes street commerce, and no other block is at zero distance from any
# register set.
own = BLOCKS.loc[BLOCKS.street_any & BLOCKS.street_known & BLOCKS.inhabited,
                 'CVEGEO_MZA']
on_own = A.CVEGEO_MZA.isin(own)
for s in STREET_SETS:
    z = A.loc[on_own, f'distance_m_{s}']
    assert (z == 0).all(), s
print(f'6. blocks on their own street commerce       {int(on_own.sum()):,}, '
      f'all at zero distance under the street sets')
zero_reg = int(sum((A[f'distance_m_{s}'] == 0).sum() for s in REGISTER_SETS))
print(f'   blocks at zero distance under a register set  {zero_reg}')

# Every measured distance is at least the block's own attachment, since the
# attachment is part of the route, except where the block is itself the
# destination.
short = 0
for s in SETS:
    v = A[f'distance_m_{s}']
    short += int(((v < A['snap_m'] - 1e-6) & (v > 0)).sum())
print(f'7. distances shorter than the block attachment {short}')
assert short == 0

# Removing destinations cannot shorten a distance, so the sets without shared
# points and the thinned sets are never closer than the sets they come from.
M = A[['CVEGEO_MZA', 'distance_m_supermarket_club', 'distance_m_fresh_only']].merge(
    ROB, on='CVEGEO_MZA')
bad = int((M['distance_m_supermarket_club_noshared']
           < M['distance_m_supermarket_club'] - 1e-6).sum())
bad += int((M['distance_m_fresh_only_noshared']
            < M['distance_m_fresh_only'] - 1e-6).sum())
for r in range(R_DRAWS):
    bad += int((M[f'distance_m_fresh_thinned_{r}']
                < M['distance_m_fresh_only'] - 1e-6).sum())
print(f'8. robustness sets closer than the sets they come from  {bad}')
assert bad == 0

print(f'\n9. population measured                       {A.POBTOT.sum():,.0f}')
print(f'10. median distance, supermarkets and clubs  '
      f'{wq(A, "distance_m_supermarket_club", 0.5):,.0f} m')
print(f'11. median distance, fresh and staple sellers '
      f'{wq(A, "distance_m_fresh_only", 0.5):,.0f} m')
print(f'12. median distance, every food source       '
      f'{wq(A, "distance_m_all_food", 0.5):,.0f} m')
print(f'13. median attachment of a block to a node   {A.snap_m.median():,.1f} m')

## 12. Write results

In [ ]:
keep = (['CVEGEO_MZA', 'slug', 'POBTOT', 'snap_m', 'x_m', 'y_m']
        + [f'distance_m_{s}' for s in SETS]
        + [f'walk_min_{s}' for s in SETS])
out = os.path.join(RESULTS, '07_block_access.parquet')
A[keep].to_parquet(out, index=False)
print('written:', out, f'({os.path.getsize(out) / 1e6:.1f} MB)')

outn = os.path.join(RESULTS, '07_network_attachment.parquet')
NET.to_parquet(outn, index=False)
print('written:', outn, f'({os.path.getsize(outn) / 1e6:.1f} MB)')

outr = os.path.join(RESULTS, '07_block_access_robustness.parquet')
ROB[['CVEGEO_MZA', 'slug', 'POBTOT']
    + [f'distance_m_{s}' for s in ROB_SETS]].to_parquet(outr, index=False)
print('written:', outr, f'({os.path.getsize(outr) / 1e6:.1f} MB)')

mx.flow('blocks in the frame', len(BLOCKS), 'from the census join')
mx.flow('inhabited blocks', int(BLOCKS.inhabited.sum()),
        'uninhabited blocks have no resident to measure')
mx.flow('blocks measured', len(A), 'block geometry found and attached')
mx.flow('blocks with a distance to every set', int(
    A[[f'distance_m_{s}' for s in SETS]].notna().all(axis=1).sum()),
        'remainder lie in an unreachable part of a network')

print()
print(mx.flow_table().to_string(index=False))
mx.save_flow(os.path.join(mx.DIRS['logs'], '07_sample_flow.csv'))
rep.add_table('attrition', mx.flow_table(), max_rows=20)

rep.note(f'{len(A):,} inhabited blocks, {A.POBTOT.sum():,.0f} people; '
         f'median walking distance '
         f'{wq(A, "distance_m_supermarket_club", 0.5):,.0f} m to a supermarket '
         f'or warehouse club, {wq(A, "distance_m_fresh_only", 0.5):,.0f} m to a '
         f'specialist fresh or staple seller, '
         f'{wq(A, "distance_m_all_food_street", 0.5):,.0f} m to any food '
         f'source of any kind')
rep.save()